# Creating Causal CelebA Dataset By Construction

We will walk through the construction of two causal CelebA datasets.

Gender <--> Age -> HairColor

Gender <--> Age -> Eyeglasses

These will be used in downstream experiments on disentangled causal representation learning

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from copy import copy
from pathlib import Path
from pprint import pprint
from collections import Counter
import re
import os

from joblib import Parallel, delayed
from tqdm import tqdm
from tqdm_joblib import tqdm_joblib

from PIL import Image
import numpy as np
from numpy.testing import assert_allclose
import pandas as pd
import torch
from torchvision.datasets import CelebA
from torchvision import transforms
from torch.utils.data import DataLoader, random_split
from torchvision.utils import save_image, make_grid
import seaborn as sns
import matplotlib.pyplot as plt

from ciflows.datasets.causalceleba_scm.sampling import get_random_transforms, exponential_weights

/Users/adam2392/miniforge3/envs/ciflows/lib/python3.11/site-packages/tqdm_joblib/__init__.py:4: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [3]:
# Root directory for the dataset
data_root = Path("/Users/adam2392/pytorch_data/")
# data_root = Path('/local/eb/adam2392')

# Spatial size of training images, images are resized to this size.
image_size = 128

celeba_data = CelebA(
    data_root,
    download=True,
    target_type="identity",
    transform=transforms.Compose(
        [
            transforms.Resize(image_size),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            # transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ]
    ),
)

Files already downloaded and verified


## Haircolor Dataset

First, let's demonstrate the preprocessing needed to create the causal dataset involving hair color.

In [4]:
attr_names = celeba_data.attr_names
gender_idx = attr_names.index("Male")
age_idx = attr_names.index("Young")

blackhair_idx = attr_names.index("Black_Hair")
blondhair_idx = attr_names.index("Blond_Hair")
brownhair_idx = attr_names.index("Brown_Hair")
grayhair_idx = attr_names.index("Gray_Hair")
print(celeba_data.attr[:, gender_idx].shape)

print(blackhair_idx, blondhair_idx, brownhair_idx, grayhair_idx)

torch.Size([162770])
8 9 11 17


In [5]:
# create a dataframe for the celebA attributes
df = pd.DataFrame(celeba_data.attr, columns=celeba_data.attr_names[:-1])
df["sample_idx"] = np.arange(len(df), dtype=int)

# now filter the dataframe based on meeting exactly one of the chosen hair colors
hair_colors = ["Black_Hair", "Blond_Hair", "Gray_Hair"]
df_filtered = df[df[hair_colors].sum(axis=1) == 1]
df_filtered.reset_index(inplace=True, drop=True)

hair_map = {"Black_Hair": 1, "Blond_Hair": 2, "Gray_Hair": 3}
df_filtered["Hair_Category"] = df_filtered[hair_colors].idxmax(axis=1).map(hair_map)

df_filtered = df_filtered.loc[:, ["sample_idx", "Male", "Young", "Hair_Category"]]

print(
    f"Number of total samples used: {len(df_filtered)} filtered from total of {len(df)} - {len(df_filtered) / len(df):.3f} of the total"
)
display(df_filtered.head())
print(df_filtered["Hair_Category"].value_counts())

Number of total samples used: 69193 filtered from total of 162770 - 0.425 of the total


/var/folders/6_/sl83qtkd68x3_mvfys07_6qm0000gn/T/ipykernel_90744/818705180.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["Hair_Category"] = df_filtered[hair_colors].idxmax(axis=1).map(hair_map)


,sample_idx,Male,Young,Hair_Category
0,6,1,1,1
1,7,1,1,1
2,10,0,1,1
3,11,1,1,1
4,12,1,1,2


Hair_Category
1    38897
2    23836
3     6460
Name: count, dtype: int64


In [39]:
def obs_sample_img_indices(
    df, male_col, young_col, hair_cat_col, hair_categories, n_samples=1000, seed=None
):
    """Set up the observational SCM."""
    rng = np.random.default_rng(seed)

    # get the index over the files
    file_sample_indices = df["sample_idx"].values

    # index over the rows in the dataframe
    male_attrs = df[male_col].values
    young_attrs = df[young_col].values
    hair_attrs = df[hair_cat_col].values

    # assert hair_attrs.shape[1] == len(hair_categories)
    assert len(file_sample_indices) == len(male_attrs) == len(young_attrs) == len(hair_attrs)
    image_attrs = np.concatenate(
        (
            file_sample_indices.reshape(-1, 1),
            male_attrs.reshape(-1, 1),
            young_attrs.reshape(-1, 1),
            hair_attrs.reshape(-1, 1),
        ),
        axis=1,
    )

    # List to store sampled indices
    sampled_indices = []
    sampled_attrs = []
    sampled_attrs_encodings = []

    gender_map = {"Male": 1, "Female": 0}
    age_map = {"Young": 1, "Old": 0}
    flag = True

    for idx in range(n_samples):
        # now, sample U_gh and use this to initialize the sampling process
        U_genderhair = rng.uniform()

        # now sample male based on bernoulli
        p_male = U_genderhair
        gender_str = "Male" if rng.uniform() < p_male else "Female"
        # gender_str = "Male" if p_male < 0.7 else "Female"
        gender = gender_map[gender_str]

        # U_genderhair = rng.uniform()
        p_old = U_genderhair
        age_str = "Old" if rng.uniform() < p_old else "Young"
        # age_str = "Old" if p_old < 0.7 else "Young"
        age = age_map[age_str]

        # sample hair
        hair_range = np.arange(len(hair_categories))
        if age_str == "Young":
            hair_range = hair_range[::-1]
        p_hairs = exponential_weights(hair_range, alpha=1.0)
        if idx == 0:
            print(hair_range)
            print(age_str, p_hairs)
        if age_str == "Young" and flag:
            print(hair_range)
            print(age_str, p_hairs)
            flag = False
        hair_str = rng.choice(hair_categories, p=p_hairs)

        # 0: black
        # 1: blond
        # 2: brown
        # 3: gray
        # hair_map = {"Black": 0, "Blond": 1, "Brown": 2, "Gray": 3}
        hair_map = {"Black": 1, "Blond": 2, "Gray": 3}
        reverse_hair_map = {v: k for k, v in hair_map.items()}

        hair = hair_map[hair_str]

        # now sample an individual that is Male, Old and X-Hair color
        matching_rows = image_attrs[
            (image_attrs[:, 1] == gender) & (image_attrs[:, 2] == age) & (image_attrs[:, 3] == hair)
        ]
        matching_row_samples = matching_rows[:, 0].tolist()  # Extract indices

        # Sample a single individual if there are matches
        if matching_row_samples:
            rand_file_index = rng.choice(matching_row_samples)
            sampled_indices.append(rand_file_index)
            sampled_attrs.append((gender_str, age_str, hair_str))
            sampled_attrs_encodings.append((gender, age, hair))
        else:
            print(f"no matching for {gender_map[gender]} {age_map[age]} {reverse_hair_map[hair]}")

    return sampled_indices, sampled_attrs, sampled_attrs_encodings


def get_joint_probability_table(sampled_attrs, verbose=False):
    # Step 1: Count occurrences of each combination
    counter = Counter(sampled_attrs)

    # Step 2: Create a DataFrame for analysis
    df = pd.DataFrame(counter.items(), columns=["Combination", "Count"])
    if verbose:
        display(df.head())
    df[["Gender", "Age", "Hair Color"]] = pd.DataFrame(df["Combination"].tolist(), index=df.index)
    df = df.drop(columns="Combination")

    # Step 3: Calculate joint probabilities
    total_count = df["Count"].sum()
    df["Joint Probability"] = df["Count"] / total_count

    # P(Gender | Age, Hair Color)
    df["P(Gender | Age, Hair Color)"] = df.groupby(["Age", "Hair Color"])["Count"].transform(
        lambda x: x / x.sum()
    )

    # P(Age | Gender, Hair Color)
    df["P(Age | Gender, Hair Color)"] = df.groupby(["Gender", "Hair Color"])["Count"].transform(
        lambda x: x / x.sum()
    )

    # P(Hair Color | Gender, Age)
    df["P(Hair Color | Gender, Age)"] = df.groupby(["Gender", "Age"])["Count"].transform(
        lambda x: x / x.sum()
    )

    # test that the output makes sense
    assert_allclose(df["Joint Probability"].sum(), 1.0)
    return df


def interventional_sample_img_indices(
    df, male_col, young_col, hair_cat_col, hair_categories, interv_idx, n_samples=1000, seed=None
):
    """Set up the observational SCM."""
    # Precompute hair categories
    if interv_idx == 0:
        hair_categories = ["Black", "Gray"]
    elif interv_idx == 1:
        hair_categories = ["Black", "Blond"]
    elif interv_idx == 2:
        hair_categories = ["Gray"]
    elif interv_idx == 3:
        hair_categories = ["Blond"]
    else:
        raise ValueError("Invalid intervention idx")

    rng = np.random.default_rng(seed)

    # get the index over the files
    file_sample_indices = df["sample_idx"].values

    # index over the rows in the dataframe
    male_attrs = df[male_col].values
    young_attrs = df[young_col].values
    hair_attrs = df[hair_cat_col].values

    # assert hair_attrs.shape[1] == len(hair_categories)
    assert len(file_sample_indices) == len(male_attrs) == len(young_attrs) == len(hair_attrs)
    image_attrs = np.concatenate(
        (
            file_sample_indices.reshape(-1, 1),
            male_attrs.reshape(-1, 1),
            young_attrs.reshape(-1, 1),
            hair_attrs.reshape(-1, 1),
        ),
        axis=1,
    )

    # List to store sampled indices
    sampled_indices = []
    sampled_attrs = []
    sampled_attrs_encodings = []

    gender_map = {"Male": 1, "Female": 0}
    age_map = {"Young": 1, "Old": 0}
    flag = True

    for idx in range(n_samples):
        # now, sample U_gh and use this to initialize the sampling process
        U_genderhair = rng.uniform()

        # now sample male based on bernoulli
        p_male = U_genderhair
        gender_str = "Male" if rng.uniform() < p_male else "Female"
        gender = gender_map[gender_str]

        p_old = U_genderhair
        age_str = "Old" if rng.uniform() < p_old else "Young"
        age = age_map[age_str]

        hair_range = np.arange(len(hair_categories))
        if age_str == "Young":
            hair_range = hair_range[::-1]
        p_hairs = exponential_weights(hair_range, alpha=1.0)
        hair_str = rng.choice(hair_categories, p=p_hairs)

        # 0: black
        # 1: blond
        # 2: brown
        # 3: gray
        hair_map = {"Black": 1, "Blond": 2, "Gray": 3}
        reverse_hair_map = {v: k for k, v in hair_map.items()}

        hair = hair_map[hair_str]

        # now sample an individual that is Male, Old and X-Hair color
        matching_rows = image_attrs[
            (image_attrs[:, 1] == gender) & (image_attrs[:, 2] == age) & (image_attrs[:, 3] == hair)
        ]
        matching_row_samples = matching_rows[:, 0].tolist()  # Extract indices

        # Sample a single individual if there are matches
        if matching_row_samples:
            rand_file_index = rng.choice(matching_row_samples)
            sampled_indices.append(rand_file_index)
            sampled_attrs.append((gender_str, age_str, hair_str))
            sampled_attrs_encodings.append((gender, age, hair))
        else:
            print(f"no matching for {gender_map[gender]} {age_map[age]} {reverse_hair_map[hair]}")

    return sampled_indices, sampled_attrs, sampled_attrs_encodings

In [40]:
df = df_filtered.copy()
n_samples = 20_000
seed = 12345
hair_cat_col = "Hair_Category"
male_col = "Male"
young_col = "Young"
# Precompute hair categories
hair_categories = ["Black", "Blond", "Gray"]

print([blackhair_idx, blondhair_idx, grayhair_idx])
sampled_indices, sampled_attrs, sampled_attrs_encodings = obs_sample_img_indices(
    df, male_col, young_col, hair_cat_col, hair_categories, n_samples=n_samples, seed=seed
)

[8, 9, 17]
[2 1 0]
Young [0.6652409557748219, 0.24472847105479764, 0.09003057317038046]
[2 1 0]
Young [0.6652409557748219, 0.24472847105479764, 0.09003057317038046]


In [41]:
print(sampled_attrs[:5])
print(sampled_attrs_encodings[:5])

[('Female', 'Young', 'Blond'), ('Female', 'Old', 'Gray'), ('Male', 'Young', 'Blond'), ('Female', 'Young', 'Black'), ('Male', 'Old', 'Blond')]
[(0, 1, 2), (0, 0, 3), (1, 1, 2), (0, 1, 1), (1, 0, 2)]


In [42]:
df = get_joint_probability_table(sampled_attrs)

# Display the resulting table
display(df)

# save resulting dataframe table
# df.to_csv(
#     data_root / "CausalCelebA" / "chain" / "dim128" / "causalceleba_obs_joint_probability.csv",
#     index=False,
# )

print(df.shape)

,Count,Gender,Age,Hair Color,Joint Probability,"P(Gender | Age, Hair Color)","P(Age | Gender, Hair Color)","P(Hair Color | Gender, Age)"
0,1599,Female,Young,Blond,0.07995,0.658567,0.665695,0.234733
1,2225,Female,Old,Gray,0.11125,0.335292,0.784003,0.666567
2,829,Male,Young,Blond,0.04145,0.341433,0.336718,0.253827
3,4600,Female,Young,Black,0.23000,0.686567,0.936864,0.675279
4,1633,Male,Old,Blond,0.08165,0.670361,0.663282,0.248026
5,2100,Male,Young,Black,0.10500,0.313433,0.795455,0.642988
6,4411,Male,Old,Gray,0.22055,0.664708,0.929023,0.669957
7,803,Female,Old,Blond,0.04015,0.329639,0.334305,0.240563
8,613,Female,Young,Gray,0.03065,0.645263,0.215997,0.089988
9,337,Male,Young,Gray,0.01685,0.354737,0.070977,0.103184


(12, 8)


In [12]:
def celeba_scm(
    celeba_data,
    save_dir,
    sample_indices,
    append=False,
    img_size=128,
    n_workers=-1,
):
    if append:
        # Define the pattern to match the file names
        pattern = re.compile(r"sample_(\d+)\.jpg")
        max_idx = 0  # Start with a default value for empty directory
        for file_name in os.listdir(save_dir):
            match = pattern.match(file_name)
            if match:
                idx = int(match.group(1))  # Extract the number
                max_idx = max(max_idx, idx)  # Update the maximum
    else:
        max_idx = 0

    def process_sample(sample_idx, idx_offset):
        # Load image and metadata
        image, _ = celeba_data[sample_idx]
        image = torch.permute(image, (1, 2, 0))

        # Apply transformations
        transform_pipeline = get_random_transforms(image_size=img_size)
        transformed = transform_pipeline(image=np.array(image))
        transformed_image = transformed["image"]

        # Convert to a PIL Image
        transformed_image = (transformed_image.numpy() * 255).astype(np.uint8)
        if transformed_image.shape[0] == 3:
            transformed_image = np.transpose(transformed_image, (1, 2, 0))
        image_pil = Image.fromarray(transformed_image)

        # Save the image as JPG
        save_path = save_dir / f"sample_{idx_offset}.jpg"
        image_pil.save(save_path)

    if n_workers == 1:
        # Sequential execution
        for idx, sample_idx in tqdm(enumerate(sample_indices), desc="Processing Samples"):
            process_sample(sample_idx, idx + max_idx)
    else:
        # Wrap Parallel jobs with tqdm
        with tqdm_joblib(tqdm(desc="Processing Samples", total=len(sample_indices))):
            Parallel(n_jobs=n_workers)(
                delayed(process_sample)(sample_idx, idx + max_idx)
                for idx, sample_idx in enumerate(sample_indices)
            )

In [11]:
scm_type = "obs"
# scm_type = "int_hair_2"
# interv_idx = 2
append = True

save_dir = data_root / "CausalCelebA" / "chain" / "dim128" / scm_type
save_dir.mkdir(exist_ok=True, parents=True)

celeba_scm(
    celeba_data,
    save_dir,
    sample_indices=sampled_indices[7700:],
    append=append,
    img_size=image_size,
    n_workers=1,
)

# save the metadata csv files
saved_causal_df = pd.DataFrame(sampled_attrs, columns=["Gender", "Age", "Hair Color"])
saved_causal_df["Sample Index"] = sampled_indices

# save the metadata csv files
if scm_type == "obs":
    saved_causal_df["Intervention"] = "Obs"
else:
    saved_causal_df["Intervention"] = "Haircolor"

causal_attrs_path = save_dir / "causal_attrs.csv"
meta_attrs_path = save_dir / "meta_attrs.csv"
if append:
    # TODO: need to append the existing CSV
    # Define file paths

    # Check if the files already exist and append if they do
    if causal_attrs_path.exists():
        existing_causal_df = pd.read_csv(causal_attrs_path, index_col=0)
        saved_causal_df = pd.concat([existing_causal_df, saved_causal_df], ignore_index=True)

saved_causal_df.to_csv(causal_attrs_path)

12300it [00:26, 466.07it/s]


# Generate Interventional Datasets with API

The above demonstrates how and what to do with the CelebA dataset to create the causal dataset involving hair color. Now we'll generate the interventional datasets.

In [13]:
df = df_filtered.copy()
n_samples = 20_000
seed = 12345
hair_cat_col = "Hair_Category"
male_col = "Male"
young_col = "Young"
# Precompute hair categories
hair_categories = ["Black", "Blond", "Gray"]

print([blackhair_idx, blondhair_idx, grayhair_idx])


scm_type = "int_hair_1"
interv_idx = 1
append = False
# sample the corresponding indices
sampled_indices, sampled_attrs, sampled_attrs_encodings = interventional_sample_img_indices(
    df,
    male_col,
    young_col,
    hair_cat_col,
    hair_categories,
    interv_idx=interv_idx,
    n_samples=n_samples,
    seed=seed,
)

# create the directory
save_dir = data_root / "CausalCelebA" / "chain" / "dim128" / scm_type
save_dir.mkdir(exist_ok=True, parents=True)

celeba_scm(
    celeba_data,
    save_dir,
    sample_indices=sampled_indices,
    append=append,
    img_size=image_size,
    n_workers=1,
)

# save the metadata csv files
saved_causal_df = pd.DataFrame(sampled_attrs, columns=["Gender", "Age", "Hair Color"])
saved_causal_df["Sample Index"] = sampled_indices

# save the metadata csv files
if scm_type == "obs":
    saved_causal_df["Intervention"] = "Obs"
else:
    saved_causal_df["Intervention"] = "Haircolor"

causal_attrs_path = save_dir / "causal_attrs.csv"
if append:
    # Check if the files already exist and append if they do
    if causal_attrs_path.exists():
        existing_causal_df = pd.read_csv(causal_attrs_path, index_col=0)
        saved_causal_df = pd.concat([existing_causal_df, saved_causal_df], ignore_index=True)

saved_causal_df.to_csv(causal_attrs_path)

[8, 9, 17]


100%|██████████| 20000/20000 [52:58<00:00,  6.29it/s], ?it/s]


In [14]:
df = df_filtered.copy()
n_samples = 20_000
seed = 12345
hair_cat_col = "Hair_Category"
male_col = "Male"
young_col = "Young"
# Precompute hair categories
hair_categories = ["Black", "Blond", "Gray"]

print([blackhair_idx, blondhair_idx, grayhair_idx])


scm_type = "int_hair_0"
interv_idx = 0
append = False
# sample the corresponding indices
sampled_indices, sampled_attrs, sampled_attrs_encodings = interventional_sample_img_indices(
    df,
    male_col,
    young_col,
    hair_cat_col,
    hair_categories,
    interv_idx=interv_idx,
    n_samples=n_samples,
    seed=seed,
)

# create the directory
save_dir = data_root / "CausalCelebA" / "chain" / "dim128" / scm_type
save_dir.mkdir(exist_ok=True, parents=True)

celeba_scm(
    celeba_data,
    save_dir,
    sample_indices=sampled_indices,
    append=append,
    img_size=image_size,
    n_workers=1,
)

# save the metadata csv files
saved_causal_df = pd.DataFrame(sampled_attrs, columns=["Gender", "Age", "Hair Color"])
saved_causal_df["Sample Index"] = sampled_indices

# save the metadata csv files
if scm_type == "obs":
    saved_causal_df["Intervention"] = "Obs"
else:
    saved_causal_df["Intervention"] = "Haircolor"

causal_attrs_path = save_dir / "causal_attrs.csv"
if append:
    # Check if the files already exist and append if they do
    if causal_attrs_path.exists():
        existing_causal_df = pd.read_csv(causal_attrs_path, index_col=0)
        saved_causal_df = pd.concat([existing_causal_df, saved_causal_df], ignore_index=True)

saved_causal_df.to_csv(causal_attrs_path)

[8, 9, 17]


100%|██████████| 20000/20000 [51:18<00:00,  6.50it/s], ?it/s]


In [18]:
df = df_filtered.copy()
n_samples = 20_000
seed = 12345
hair_cat_col = "Hair_Category"
male_col = "Male"
young_col = "Young"
# Precompute hair categories
hair_categories = ["Black", "Blond", "Gray"]

print([blackhair_idx, blondhair_idx, grayhair_idx])


scm_type = "int_hair_2"
interv_idx = 2
append = False
# sample the corresponding indices
sampled_indices, sampled_attrs, sampled_attrs_encodings = interventional_sample_img_indices(
    df,
    male_col,
    young_col,
    hair_cat_col,
    hair_categories,
    interv_idx=interv_idx,
    n_samples=n_samples,
    seed=seed,
)

# create the directory
save_dir = data_root / "CausalCelebA" / "chain" / "dim128" / scm_type
save_dir.mkdir(exist_ok=True, parents=True)

celeba_scm(
    celeba_data,
    save_dir,
    sample_indices=sampled_indices,
    append=append,
    img_size=image_size,
    n_workers=1,
)

# save the metadata csv files
saved_causal_df = pd.DataFrame(sampled_attrs, columns=["Gender", "Age", "Hair Color"])
saved_causal_df["Sample Index"] = sampled_indices

# save the metadata csv files
if scm_type == "obs":
    saved_causal_df["Intervention"] = "Obs"
else:
    saved_causal_df["Intervention"] = "Haircolor"

causal_attrs_path = save_dir / "causal_attrs.csv"
if append:
    # Check if the files already exist and append if they do
    if causal_attrs_path.exists():
        existing_causal_df = pd.read_csv(causal_attrs_path, index_col=0)
        saved_causal_df = pd.concat([existing_causal_df, saved_causal_df], ignore_index=True)

saved_causal_df.to_csv(causal_attrs_path)

[8, 9, 17]


Processing Samples: 20000it [00:37, 540.02it/s]


In [15]:
df = df_filtered.copy()
n_samples = 20_000
seed = 12345
hair_cat_col = "Hair_Category"
male_col = "Male"
young_col = "Young"
eyeglass_col = "Eyeglasses"
# Precompute hair categories
hair_categories = ["Black", "Blond", "Gray"]

print([blackhair_idx, blondhair_idx, grayhair_idx])


scm_type = "int_hair_3"
interv_idx = 3
append = False
# sample the corresponding indices
sampled_indices, sampled_attrs, sampled_attrs_encodings = interventional_sample_img_indices(
    df,
    male_col,
    young_col,
    hair_cat_col,
    hair_categories,
    interv_idx=interv_idx,
    n_samples=n_samples,
    seed=seed,
)

# create the directory
save_dir = data_root / "CausalCelebA" / "chain" / "dim128" / scm_type
save_dir.mkdir(exist_ok=True, parents=True)

celeba_scm(
    celeba_data,
    save_dir,
    sample_indices=sampled_indices,
    append=append,
    img_size=image_size,
    n_workers=1,
)

# save the metadata csv files
saved_causal_df = pd.DataFrame(sampled_attrs, columns=["Gender", "Age", "Hair Color"])
saved_causal_df["Sample Index"] = sampled_indices

# save the metadata csv files
if scm_type == "obs":
    saved_causal_df["Intervention"] = "Obs"
else:
    saved_causal_df["Intervention"] = "Haircolor"

causal_attrs_path = save_dir / "causal_attrs.csv"
if append:
    # Check if the files already exist and append if they do
    if causal_attrs_path.exists():
        existing_causal_df = pd.read_csv(causal_attrs_path, index_col=0)
        saved_causal_df = pd.concat([existing_causal_df, saved_causal_df], ignore_index=True)

saved_causal_df.to_csv(causal_attrs_path)

[8, 9, 17]


100%|██████████| 20000/20000 [51:31<00:00,  6.47it/s], ?it/s]


## Inspect Conditional Distributions over Causal Factors

In [19]:
from scipy.stats import chi2_contingency


# Function to compute Cramér's V
def cramers_v(confusion_matrix):
    # Perform the chi-squared test
    chi2, _, _, _ = chi2_contingency(confusion_matrix)
    n = confusion_matrix.to_numpy().sum()  # Total number of observations
    r, k = confusion_matrix.shape  # Rows and columns
    # Compute and return Cramér's V as a scalar
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))


# Function to compute pairwise correlations
def pairwise_cramers_v(df, columns):
    # Ensure the DataFrame has correct dimensions and numeric dtype
    results = pd.DataFrame(index=columns, columns=columns, dtype=float)
    for col1 in columns:
        for col2 in columns:
            if col1 == col2:
                results.at[col1, col2] = 1.0  # Correlation with itself
            else:
                # Create a contingency table
                contingency_table = pd.crosstab(df[col1], df[col2])
                corr_vals = cramers_v(contingency_table)
                # print(f"Contingency table for {col1} vs {col2}:\n{contingency_table}\n")
                # print(corr_vals)
                results.at[col1, col2] = corr_vals
    return results


# Function to compute conditional Cramér's V
def conditional_cramers_v(df, x_col, y_col, z_col):
    results = {}
    for z_value, subset in df.groupby(z_col):
        contingency_table = pd.crosstab(subset[x_col], subset[y_col])
        results[z_value] = cramers_v(contingency_table)
    return results

In [20]:
def inspect_sampled_causal_distr(df):
    # Define the columns for pairwise analysis
    # columns = ['Male', 'Young', 'Hair_Category']
    columns = ["Gender", "Age", "Hair Color"]
    df_selected = df.loc[:, columns]

    # Compute pairwise Cramér's V
    correlation_matrix = pairwise_cramers_v(df_selected, columns)

    # Display the correlation matrix
    print("\nPairwise Cramér's V Correlation Matrix:")
    display(correlation_matrix)

    # Compute conditional Cramér's V
    conditional_results = conditional_cramers_v(
        df_selected, x_col="Gender", y_col="Age", z_col="Hair Color"
    )
    print("Conditional Cramér's V:")
    for val, v in conditional_results.items():
        print(f"Hair Color = {val}: Cramér's V = {v:.4f}")

    # Compute conditional Cramér's V
    conditional_results = conditional_cramers_v(
        df_selected, z_col="Gender", x_col="Age", y_col="Hair Color"
    )
    print("Conditional Cramér's V:")
    for val, v in conditional_results.items():
        print(f"Gender = {val}: Cramér's V = {v:.4f}")

    conditional_results = conditional_cramers_v(
        df_selected, x_col="Gender", y_col="Hair Color", z_col="Age"
    )
    print("Conditional Cramér's V:")
    for val, v in conditional_results.items():
        print(f"Age = {val}: Cramér's V = {v:.4f}")

In [ ]:
scm_type = "obs"
sampled_attr_fname = data_root / "CausalCelebA" / "chain" / "dim128" / scm_type
sampled_attr_fname = sampled_attr_fname / "causal_attrs.csv"

df = pd.read_csv(sampled_attr_fname)
df.head()

inspect_sampled_causal_distr(df)

In [ ]:
scm_type = "int_hair_0"
sampled_attr_fname = data_root / "CausalCelebA" / "chain" / "dim128" / scm_type
sampled_attr_fname = sampled_attr_fname / "causal_attrs.csv"

df = pd.read_csv(sampled_attr_fname)
df.head()

print(df.shape)
inspect_sampled_causal_distr(df)

# Convert to list of tuples
sampled_attrs = [
    tuple(row)
    for row in df.loc[:, ["Gender", "Age", "Hair Color"]].itertuples(index=False, name=None)
]

print(sampled_attrs[:5])
joint_df = get_joint_probability_table(sampled_attrs, verbose=False)

# Display the resulting table
display(joint_df)

# save resulting dataframe table
fname = (
    data_root
    / "CausalCelebA"
    / "chain"
    / "dim128"
    / f"causalceleba_{scm_type}_joint_probability.csv"
)
print("Saving to ", fname)
joint_df.to_csv(
    fname,
    index=False,
)

(20000, 6)

Pairwise Cramér's V Correlation Matrix:


,Gender,Age,Hair Color
Gender,1.000000,0.339432,0.169812
Age,0.339432,1.000000,0.462810
Hair Color,0.169812,0.462810,1.000000


Conditional Cramér's V:
Hair Color = Black: Cramér's V = 0.3069
Hair Color = Gray: Cramér's V = 0.2900
Conditional Cramér's V:
Gender = Female: Cramér's V = 0.4449
Gender = Male: Cramér's V = 0.4288
Conditional Cramér's V:
Age = Old: Cramér's V = 0.0055
Age = Young: Cramér's V = 0.0245
[('Female', 'Young', 'Black'), ('Female', 'Old', 'Gray'), ('Male', 'Young', 'Black'), ('Female', 'Young', 'Black'), ('Male', 'Old', 'Black')]


,Count,Gender,Age,Hair Color,Joint Probability,"P(Gender | Age, Hair Color)","P(Age | Gender, Hair Color)","P(Hair Color | Gender, Age)"
0,5024,Female,Young,Black,0.25120,0.682980,0.847646,0.737522
1,2435,Female,Old,Gray,0.12175,0.334800,0.576604,0.729479
2,2332,Male,Young,Black,0.11660,0.317020,0.571849,0.714023
3,1746,Male,Old,Black,0.08730,0.659117,0.428151,0.265188
4,934,Male,Young,Gray,0.04670,0.343130,0.161816,0.285977
5,4838,Male,Old,Gray,0.24190,0.665200,0.838184,0.734812
6,903,Female,Old,Black,0.04515,0.340883,0.152354,0.270521
7,1788,Female,Young,Gray,0.08940,0.656870,0.423396,0.262478


Saving to  /Users/adam2392/pytorch_data/CausalCelebA/chain/dim128/causalceleba_int_hair_0_joint_probability.csv


In [46]:
scm_type = "int_hair_1"
sampled_attr_fname = data_root / "CausalCelebA" / "chain" / "dim128" / scm_type
sampled_attr_fname = sampled_attr_fname / "causal_attrs.csv"

df = pd.read_csv(sampled_attr_fname)
df.head()

inspect_sampled_causal_distr(df)


Pairwise Cramér's V Correlation Matrix:


,Gender,Age,Hair Color
Gender,1.000000,0.339432,0.169812
Age,0.339432,1.000000,0.462810
Hair Color,0.169812,0.462810,1.000000


Conditional Cramér's V:
Hair Color = Black: Cramér's V = 0.3069
Hair Color = Blond: Cramér's V = 0.2900
Conditional Cramér's V:
Gender = Female: Cramér's V = 0.4449
Gender = Male: Cramér's V = 0.4288
Conditional Cramér's V:
Age = Old: Cramér's V = 0.0055
Age = Young: Cramér's V = 0.0245


In [47]:
# Convert to list of tuples
sampled_attrs = [
    tuple(row)
    for row in df.loc[:, ["Gender", "Age", "Hair Color"]].itertuples(index=False, name=None)
]

print(sampled_attrs[:5])
joint_df = get_joint_probability_table(sampled_attrs, verbose=False)

# Display the resulting table
display(joint_df)

# save resulting dataframe table
fname = (
    data_root
    / "CausalCelebA"
    / "chain"
    / "dim128"
    / f"causalceleba_{scm_type}_joint_probability.csv"
)
print("Saving to ", fname)
joint_df.to_csv(
    fname,
    index=False,
)

[('Female', 'Young', 'Black'), ('Female', 'Old', 'Blond'), ('Male', 'Young', 'Black'), ('Female', 'Young', 'Black'), ('Male', 'Old', 'Black')]


,Count,Gender,Age,Hair Color,Joint Probability,"P(Gender | Age, Hair Color)","P(Age | Gender, Hair Color)","P(Hair Color | Gender, Age)"
0,5024,Female,Young,Black,0.25120,0.682980,0.847646,0.737522
1,2435,Female,Old,Blond,0.12175,0.334800,0.576604,0.729479
2,2332,Male,Young,Black,0.11660,0.317020,0.571849,0.714023
3,1746,Male,Old,Black,0.08730,0.659117,0.428151,0.265188
4,934,Male,Young,Blond,0.04670,0.343130,0.161816,0.285977
5,4838,Male,Old,Blond,0.24190,0.665200,0.838184,0.734812
6,903,Female,Old,Black,0.04515,0.340883,0.152354,0.270521
7,1788,Female,Young,Blond,0.08940,0.656870,0.423396,0.262478


Saving to  /Users/adam2392/pytorch_data/CausalCelebA/chain/dim128/causalceleba_int_hair_1_joint_probability.csv


In [48]:
scm_type = "int_hair_2"
sampled_attr_fname = data_root / "CausalCelebA" / "chain" / "dim128" / scm_type
sampled_attr_fname = sampled_attr_fname / "causal_attrs.csv"

df = pd.read_csv(sampled_attr_fname)
df.head()

inspect_sampled_causal_distr(df)

# Convert to list of tuples
sampled_attrs = [
    tuple(row)
    for row in df.loc[:, ["Gender", "Age", "Hair Color"]].itertuples(index=False, name=None)
]

print(sampled_attrs[:5])
joint_df = get_joint_probability_table(sampled_attrs, verbose=False)

# Display the resulting table
display(joint_df)

# save resulting dataframe table
fname = (
    data_root
    / "CausalCelebA"
    / "chain"
    / "dim128"
    / f"causalceleba_{scm_type}_joint_probability.csv"
)
print("Saving to ", fname)
joint_df.to_csv(
    fname,
    index=False,
)


Pairwise Cramér's V Correlation Matrix:


/var/folders/6_/sl83qtkd68x3_mvfys07_6qm0000gn/T/ipykernel_90744/2820289783.py:11: RuntimeWarning: invalid value encountered in divide
  return np.sqrt(chi2 / (n * (min(r, k) - 1)))
/var/folders/6_/sl83qtkd68x3_mvfys07_6qm0000gn/T/ipykernel_90744/2820289783.py:11: RuntimeWarning: invalid value encountered in divide
  return np.sqrt(chi2 / (n * (min(r, k) - 1)))
/var/folders/6_/sl83qtkd68x3_mvfys07_6qm0000gn/T/ipykernel_90744/2820289783.py:11: RuntimeWarning: invalid value encountered in divide
  return np.sqrt(chi2 / (n * (min(r, k) - 1)))
/var/folders/6_/sl83qtkd68x3_mvfys07_6qm0000gn/T/ipykernel_90744/2820289783.py:11: RuntimeWarning: invalid value encountered in divide
  return np.sqrt(chi2 / (n * (min(r, k) - 1)))


,Gender,Age,Hair Color
Gender,1.000000,0.339432,NaN
Age,0.339432,1.000000,NaN
Hair Color,NaN,NaN,1.0


Conditional Cramér's V:
Hair Color = Gray: Cramér's V = 0.3394
Conditional Cramér's V:
Gender = Female: Cramér's V = nan
Gender = Male: Cramér's V = nan
Conditional Cramér's V:
Age = Old: Cramér's V = nan
Age = Young: Cramér's V = nan
[('Female', 'Young', 'Gray'), ('Female', 'Old', 'Gray'), ('Male', 'Young', 'Gray'), ('Female', 'Young', 'Gray'), ('Male', 'Old', 'Gray')]


/var/folders/6_/sl83qtkd68x3_mvfys07_6qm0000gn/T/ipykernel_90744/2820289783.py:11: RuntimeWarning: invalid value encountered in divide
  return np.sqrt(chi2 / (n * (min(r, k) - 1)))
/var/folders/6_/sl83qtkd68x3_mvfys07_6qm0000gn/T/ipykernel_90744/2820289783.py:11: RuntimeWarning: invalid value encountered in divide
  return np.sqrt(chi2 / (n * (min(r, k) - 1)))
/var/folders/6_/sl83qtkd68x3_mvfys07_6qm0000gn/T/ipykernel_90744/2820289783.py:11: RuntimeWarning: invalid value encountered in divide
  return np.sqrt(chi2 / (n * (min(r, k) - 1)))
/var/folders/6_/sl83qtkd68x3_mvfys07_6qm0000gn/T/ipykernel_90744/2820289783.py:11: RuntimeWarning: invalid value encountered in divide
  return np.sqrt(chi2 / (n * (min(r, k) - 1)))


,Count,Gender,Age,Hair Color,Joint Probability,"P(Gender | Age, Hair Color)","P(Age | Gender, Hair Color)","P(Hair Color | Gender, Age)"
0,6812,Female,Young,Gray,0.3406,0.675928,0.671133,1.0
1,3338,Female,Old,Gray,0.1669,0.336424,0.328867,1.0
2,3266,Male,Young,Gray,0.1633,0.324072,0.331574,1.0
3,6584,Male,Old,Gray,0.3292,0.663576,0.668426,1.0


Saving to  /Users/adam2392/pytorch_data/CausalCelebA/chain/dim128/causalceleba_int_hair_2_joint_probability.csv


In [49]:
scm_type = "int_hair_3"
sampled_attr_fname = data_root / "CausalCelebA" / "chain" / "dim128" / scm_type
sampled_attr_fname = sampled_attr_fname / "causal_attrs.csv"

df = pd.read_csv(sampled_attr_fname)
df.head()

inspect_sampled_causal_distr(df)

# Convert to list of tuples
sampled_attrs = [
    tuple(row)
    for row in df.loc[:, ["Gender", "Age", "Hair Color"]].itertuples(index=False, name=None)
]

print(sampled_attrs[:5])
joint_df = get_joint_probability_table(sampled_attrs, verbose=False)

# Display the resulting table
display(joint_df)

# save resulting dataframe table
fname = (
    data_root
    / "CausalCelebA"
    / "chain"
    / "dim128"
    / f"causalceleba_{scm_type}_joint_probability.csv"
)
print("Saving to ", fname)
joint_df.to_csv(
    fname,
    index=False,
)


Pairwise Cramér's V Correlation Matrix:


/var/folders/6_/sl83qtkd68x3_mvfys07_6qm0000gn/T/ipykernel_90744/2820289783.py:11: RuntimeWarning: invalid value encountered in divide
  return np.sqrt(chi2 / (n * (min(r, k) - 1)))
/var/folders/6_/sl83qtkd68x3_mvfys07_6qm0000gn/T/ipykernel_90744/2820289783.py:11: RuntimeWarning: invalid value encountered in divide
  return np.sqrt(chi2 / (n * (min(r, k) - 1)))
/var/folders/6_/sl83qtkd68x3_mvfys07_6qm0000gn/T/ipykernel_90744/2820289783.py:11: RuntimeWarning: invalid value encountered in divide
  return np.sqrt(chi2 / (n * (min(r, k) - 1)))
/var/folders/6_/sl83qtkd68x3_mvfys07_6qm0000gn/T/ipykernel_90744/2820289783.py:11: RuntimeWarning: invalid value encountered in divide
  return np.sqrt(chi2 / (n * (min(r, k) - 1)))


,Gender,Age,Hair Color
Gender,1.000000,0.339432,NaN
Age,0.339432,1.000000,NaN
Hair Color,NaN,NaN,1.0


Conditional Cramér's V:
Hair Color = Blond: Cramér's V = 0.3394
Conditional Cramér's V:
Gender = Female: Cramér's V = nan
Gender = Male: Cramér's V = nan
Conditional Cramér's V:
Age = Old: Cramér's V = nan
Age = Young: Cramér's V = nan
[('Female', 'Young', 'Blond'), ('Female', 'Old', 'Blond'), ('Male', 'Young', 'Blond'), ('Female', 'Young', 'Blond'), ('Male', 'Old', 'Blond')]


/var/folders/6_/sl83qtkd68x3_mvfys07_6qm0000gn/T/ipykernel_90744/2820289783.py:11: RuntimeWarning: invalid value encountered in divide
  return np.sqrt(chi2 / (n * (min(r, k) - 1)))
/var/folders/6_/sl83qtkd68x3_mvfys07_6qm0000gn/T/ipykernel_90744/2820289783.py:11: RuntimeWarning: invalid value encountered in divide
  return np.sqrt(chi2 / (n * (min(r, k) - 1)))
/var/folders/6_/sl83qtkd68x3_mvfys07_6qm0000gn/T/ipykernel_90744/2820289783.py:11: RuntimeWarning: invalid value encountered in divide
  return np.sqrt(chi2 / (n * (min(r, k) - 1)))
/var/folders/6_/sl83qtkd68x3_mvfys07_6qm0000gn/T/ipykernel_90744/2820289783.py:11: RuntimeWarning: invalid value encountered in divide
  return np.sqrt(chi2 / (n * (min(r, k) - 1)))


,Count,Gender,Age,Hair Color,Joint Probability,"P(Gender | Age, Hair Color)","P(Age | Gender, Hair Color)","P(Hair Color | Gender, Age)"
0,6812,Female,Young,Blond,0.3406,0.675928,0.671133,1.0
1,3338,Female,Old,Blond,0.1669,0.336424,0.328867,1.0
2,3266,Male,Young,Blond,0.1633,0.324072,0.331574,1.0
3,6584,Male,Old,Blond,0.3292,0.663576,0.668426,1.0


Saving to  /Users/adam2392/pytorch_data/CausalCelebA/chain/dim128/causalceleba_int_hair_3_joint_probability.csv


# Generate Multidistributional Dataset of Gender, Age, EyeGlasses

In [56]:
n_samples = 20_000
seed = 12345
male_col = "Male"
young_col = "Young"
eyeglass_col = "Eyeglasses"

In [57]:
# create a dataframe for the celebA attributes
df = pd.DataFrame(celeba_data.attr, columns=celeba_data.attr_names[:-1])
df["sample_idx"] = np.arange(len(df), dtype=int)

# now filter the dataframe based on meeting exactly one of the chosen hair colors
# hair_colors = ["Black_Hair", "Blond_Hair", "Gray_Hair"]
# df_filtered = df[df[hair_colors].sum(axis=1) == 1]
# df_filtered.reset_index(inplace=True, drop=True)

# hair_map = {"Black_Hair": 1, "Blond_Hair": 2, "Gray_Hair": 3}
# df_filtered["Hair_Category"] = df_filtered[hair_colors].idxmax(axis=1).map(hair_map)

# df_filtered = df_filtered.loc[:, ["sample_idx", "Male", "Young", "Hair_Category"]]

print(f"Number of total samples used: {len(df)}")
display(df.head())
# print(df_filtered["Hair_Category"].value_counts())

Number of total samples used: 162770


,5_o_Clock_Shadow,Arched_Eyebrows,Attractive,Bags_Under_Eyes,Bald,Bangs,Big_Lips,Big_Nose,Black_Hair,Blond_Hair,...,Smiling,Straight_Hair,Wavy_Hair,Wearing_Earrings,Wearing_Hat,Wearing_Lipstick,Wearing_Necklace,Wearing_Necktie,Young,sample_idx
0,0,1,1,0,0,0,0,0,0,0,...,1,1,0,1,0,1,0,0,1,0
1,0,0,0,1,0,0,0,1,0,0,...,1,0,0,0,0,0,0,0,1,1
2,0,0,0,0,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,1,2
3,0,0,1,0,0,0,0,0,0,0,...,0,1,0,1,0,1,1,0,1,3
4,0,1,1,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,1,4


In [63]:
class CelebAGlassesSCM:
    @staticmethod
    def obs_sample_img_indices(df, male_col, young_col, eyeglass_col, n_samples=1000, seed=None):
        """Set up the observational SCM."""
        rng = np.random.default_rng(seed)
        eyeglass_prob = 0.8

        # get the index over the files
        file_sample_indices = df["sample_idx"].values

        # index over the rows in the dataframe
        male_attrs = df[male_col].values
        young_attrs = df[young_col].values
        eyeglass_attrs = df[eyeglass_col].values

        assert (
            len(file_sample_indices) == len(male_attrs) == len(young_attrs) == len(eyeglass_attrs)
        )
        image_attrs = np.concatenate(
            (
                file_sample_indices.reshape(-1, 1),
                male_attrs.reshape(-1, 1),
                young_attrs.reshape(-1, 1),
                eyeglass_attrs.reshape(-1, 1),
            ),
            axis=1,
        )

        # List to store sampled indices
        sampled_indices = []
        sampled_attrs = []
        sampled_attrs_encodings = []

        gender_map = {"Male": 1, "Female": 0}
        age_map = {"Young": 1, "Old": 0}
        flag = True

        for idx in range(n_samples):
            # now, sample U_gh and use this to initialize the sampling process
            U_genderhair = rng.uniform()

            # now sample male based on bernoulli
            p_male = U_genderhair
            gender_str = "Male" if rng.uniform() < p_male else "Female"
            # gender_str = "Male" if p_male < 0.7 else "Female"
            gender = gender_map[gender_str]

            # U_genderhair = rng.uniform()
            p_old = U_genderhair
            age_str = "Old" if rng.uniform() < p_old else "Young"
            # age_str = "Old" if p_old < 0.7 else "Young"
            age = age_map[age_str]

            # sample eyeglasses
            p_eyeglass = eyeglass_prob if age == 0 else 1 - eyeglass_prob
            eyeglass = 1 if rng.uniform() < p_eyeglass else 0
            eyeglass_str = "Eyeglass" if eyeglass == 1 else "No Glasses"
            if idx == 0:
                print(p_eyeglass, age_str, eyeglass)
            if age_str == "Young" and flag:
                print(age_str, p_eyeglass)
                flag = False

            # now sample an individual that is Male, Old and X-Hair color
            matching_rows = image_attrs[
                (image_attrs[:, 1] == gender)
                & (image_attrs[:, 2] == age)
                & (image_attrs[:, 3] == eyeglass)
            ]
            matching_row_samples = matching_rows[:, 0].tolist()  # Extract indices

            # Sample a single individual if there are matches
            if matching_row_samples:
                rand_file_index = rng.choice(matching_row_samples)
                sampled_indices.append(rand_file_index)
                sampled_attrs.append((gender_str, age_str, eyeglass_str))
                sampled_attrs_encodings.append((gender, age, eyeglass))
            else:
                print(f"no matching for {gender_map[gender]} {age_map[age]}")

        return sampled_indices, sampled_attrs, sampled_attrs_encodings

    @staticmethod
    def get_joint_probability_table(sampled_attrs, verbose=False):
        # Step 1: Count occurrences of each combination
        counter = Counter(sampled_attrs)

        # Step 2: Create a DataFrame for analysis
        df = pd.DataFrame(counter.items(), columns=["Combination", "Count"])
        if verbose:
            display(df.head())
        df[["Gender", "Age", "Eye Glasses"]] = pd.DataFrame(
            df["Combination"].tolist(), index=df.index
        )
        df = df.drop(columns="Combination")

        # Step 3: Calculate joint probabilities
        total_count = df["Count"].sum()
        df["Joint Probability"] = df["Count"] / total_count

        # P(Gender | Age, Hair Color)
        df["P(Gender | Age, Eye Glasses)"] = df.groupby(["Age", "Eye Glasses"])["Count"].transform(
            lambda x: x / x.sum()
        )

        # P(Age | Gender, Hair Color)
        df["P(Age | Gender, Eye Glasses)"] = df.groupby(["Gender", "Eye Glasses"])[
            "Count"
        ].transform(lambda x: x / x.sum())

        # P(Hair Color | Gender, Age)
        df["P(Eye Glasses | Gender, Age)"] = df.groupby(["Gender", "Age"])["Count"].transform(
            lambda x: x / x.sum()
        )

        # test that the output makes sense
        assert_allclose(df["Joint Probability"].sum(), 1.0)
        return df

    @staticmethod
    def interventional_sample_img_indices(
        df, male_col, young_col, eyeglass_col, interv_idx, n_samples=1000, seed=None
    ):
        """Set up the observational SCM."""
        # Precompute hair categories
        if interv_idx == 0:
            eyeglass_prob = 0.95
        elif interv_idx == 1:
            eyeglass_prob = 0.1
        else:
            raise ValueError("Invalid intervention idx")

        rng = np.random.default_rng(seed)

        # get the index over the files
        file_sample_indices = df["sample_idx"].values

        # index over the rows in the dataframe
        male_attrs = df[male_col].values
        young_attrs = df[young_col].values
        eyeglass_attrs = df[eyeglass_col].values

        assert (
            len(file_sample_indices) == len(male_attrs) == len(young_attrs) == len(eyeglass_attrs)
        )
        image_attrs = np.concatenate(
            (
                file_sample_indices.reshape(-1, 1),
                male_attrs.reshape(-1, 1),
                young_attrs.reshape(-1, 1),
                eyeglass_attrs.reshape(-1, 1),
            ),
            axis=1,
        )

        # List to store sampled indices
        sampled_indices = []
        sampled_attrs = []
        sampled_attrs_encodings = []

        gender_map = {"Male": 1, "Female": 0}
        age_map = {"Young": 1, "Old": 0}
        flag = True

        for idx in range(n_samples):
            # now, sample U_gh and use this to initialize the sampling process
            U_genderhair = rng.uniform()

            # now sample male based on bernoulli
            p_male = U_genderhair
            gender_str = "Male" if rng.uniform() < p_male else "Female"
            # gender_str = "Male" if p_male < 0.7 else "Female"
            gender = gender_map[gender_str]

            # U_genderhair = rng.uniform()
            p_old = U_genderhair
            age_str = "Old" if rng.uniform() < p_old else "Young"
            # age_str = "Old" if p_old < 0.7 else "Young"
            age = age_map[age_str]

            # sample eyeglasses
            p_eyeglass = eyeglass_prob if age == 0 else 1 - eyeglass_prob
            eyeglass = 1 if rng.uniform() < p_eyeglass else 0
            eyeglass_str = "Eyeglass" if eyeglass == 1 else "No Glasses"
            if idx == 0:
                print(p_eyeglass, age_str, eyeglass)
            if age_str == "Young" and flag:
                print(age_str, p_eyeglass)
                flag = False

            # now sample an individual that is Male, Old and X-Hair color
            matching_rows = image_attrs[
                (image_attrs[:, 1] == gender)
                & (image_attrs[:, 2] == age)
                & (image_attrs[:, 3] == eyeglass)
            ]
            matching_row_samples = matching_rows[:, 0].tolist()  # Extract indices

            # Sample a single individual if there are matches
            if matching_row_samples:
                rand_file_index = rng.choice(matching_row_samples)
                sampled_indices.append(rand_file_index)
                sampled_attrs.append((gender_str, age_str, eyeglass_str))
                sampled_attrs_encodings.append((gender, age, eyeglass))
            else:
                print(f"no matching for {gender_map[gender]} {age_map[age]}")

        return sampled_indices, sampled_attrs, sampled_attrs_encodings

    @staticmethod
    def inspect_sampled_causal_distr(df):
        # Define the columns for pairwise analysis
        # columns = ['Male', 'Young', 'Hair_Category']
        columns = ["Gender", "Age", "Eyeglasses"]
        df_selected = df.loc[:, columns]

        # Compute pairwise Cramér's V
        correlation_matrix = pairwise_cramers_v(df_selected, columns)

        # Display the correlation matrix
        print("\nPairwise Cramér's V Correlation Matrix:")
        display(correlation_matrix)

        # Compute conditional Cramér's V
        conditional_results = conditional_cramers_v(
            df_selected, x_col="Gender", y_col="Age", z_col="Eyeglasses"
        )
        print("Conditional Cramér's V:")
        for val, v in conditional_results.items():
            print(f"Eyeglasses = {val}: Cramér's V = {v:.4f}")

        # Compute conditional Cramér's V
        conditional_results = conditional_cramers_v(
            df_selected, z_col="Gender", x_col="Age", y_col="Eyeglasses"
        )
        print("Conditional Cramér's V:")
        for val, v in conditional_results.items():
            print(f"Gender = {val}: Cramér's V = {v:.4f}")

        conditional_results = conditional_cramers_v(
            df_selected, x_col="Gender", y_col="Eyeglasses", z_col="Age"
        )
        print("Conditional Cramér's V:")
        for val, v in conditional_results.items():
            print(f"Age = {val}: Cramér's V = {v:.4f}")

In [60]:
append = True

scm_types = [
    # 'obs',
    "int_eye_0",
    "int_eye_1",
]
interv_indices = [
    # None,
    0,
    1,
]
for scm_type, interv_idx in zip(scm_types, interv_indices):

    if scm_type == "obs":
        sampled_indices, sampled_attrs, sampled_attrs_encodings = (
            CelebAGlassesSCM.obs_sample_img_indices(
                df=df,
                male_col=male_col,
                young_col=young_col,
                eyeglass_col=eyeglass_col,
                n_samples=n_samples,
                seed=seed,
            )
        )
    else:
        # sample the corresponding indices
        sampled_indices, sampled_attrs, sampled_attrs_encodings = (
            CelebAGlassesSCM.interventional_sample_img_indices(
                df,
                male_col,
                young_col,
                eyeglass_col=eyeglass_col,
                interv_idx=interv_idx,
                n_samples=n_samples,
                seed=seed,
            )
        )

    save_dir = data_root / "CausalCelebA" / "eyeglass" / "dim128" / scm_type
    save_dir.mkdir(exist_ok=True, parents=True)

    celeba_scm(
        celeba_data,
        save_dir,
        sample_indices=sampled_indices,
        append=append,
        img_size=image_size,
        n_workers=1,
    )

    # save the metadata csv files
    saved_causal_df = pd.DataFrame(sampled_attrs, columns=["Gender", "Age", "Eyeglasses"])
    saved_causal_df["Sample Index"] = sampled_indices

    # save the metadata csv files
    if scm_type == "obs":
        saved_causal_df["Intervention"] = "Obs"
    else:
        saved_causal_df["Intervention"] = "Eyeglasses"

    causal_attrs_path = save_dir / "causal_attrs.csv"
    if append:
        # Check if the files already exist and append if they do
        if causal_attrs_path.exists():
            existing_causal_df = pd.read_csv(causal_attrs_path, index_col=0)
            saved_causal_df = pd.concat([existing_causal_df, saved_causal_df], ignore_index=True)

    saved_causal_df.to_csv(causal_attrs_path)

0.050000000000000044 Young 0
Young 0.050000000000000044


Processing Samples: 20000it [00:38, 516.42it/s]


0.9 Young 1
Young 0.9


Processing Samples: 20000it [00:36, 545.45it/s]


In [64]:
scm_types = ["obs", "int_eye_0", "int_eye_1"]
interv_indices = [None, 0, 1]

for scm_type, interv_idx in zip(scm_types, interv_indices):
    save_dir = data_root / "CausalCelebA" / "eyeglass" / "dim128" / scm_type
    save_dir.mkdir(exist_ok=True, parents=True)

    sampled_attr_fname = save_dir / "causal_attrs.csv"
    df = pd.read_csv(sampled_attr_fname)
    df.head()

    CelebAGlassesSCM.inspect_sampled_causal_distr(df)

    # Convert to list of tuples
    sampled_attrs = [
        tuple(row)
        for row in df.loc[:, ["Gender", "Age", "Eyeglasses"]].itertuples(index=False, name=None)
    ]

    print(sampled_attrs[:5])
    joint_df = CelebAGlassesSCM.get_joint_probability_table(sampled_attrs, verbose=False)

    # Display the resulting table
    display(joint_df)

    # save resulting dataframe table
    fname = save_dir.parent / f"causalceleba_{scm_type}_joint_probability.csv"
    print("Saving to ", fname)
    joint_df.to_csv(
        fname,
        index=False,
    )


Pairwise Cramér's V Correlation Matrix:


,Gender,Age,Eyeglasses
Gender,1.000000,0.339432,0.197744
Age,0.339432,1.000000,0.597467
Eyeglasses,0.197744,0.597467,1.000000


Conditional Cramér's V:
Eyeglasses = Eyeglass: Cramér's V = 0.2724
Eyeglasses = No Glasses: Cramér's V = 0.2900
Conditional Cramér's V:
Gender = Female: Cramér's V = 0.5825
Gender = Male: Cramér's V = 0.5674
Conditional Cramér's V:
Age = Old: Cramér's V = 0.0149
Age = Young: Cramér's V = 0.0015
[('Female', 'Young', 'No Glasses'), ('Female', 'Old', 'Eyeglass'), ('Male', 'Young', 'No Glasses'), ('Female', 'Young', 'No Glasses'), ('Male', 'Old', 'Eyeglass')]


,Count,Gender,Age,Eye Glasses,Joint Probability,"P(Gender | Age, Eye Glasses)","P(Age | Gender, Eye Glasses)","P(Eye Glasses | Gender, Age)"
0,5473,Female,Young,No Glasses,0.27365,0.676347,0.892822,0.803435
1,2681,Female,Old,Eyeglass,0.13405,0.340056,0.666915,0.803176
2,2619,Male,Young,No Glasses,0.13095,0.323653,0.654750,0.801898
3,5203,Male,Old,Eyeglass,0.26015,0.659944,0.889402,0.790249
4,647,Male,Young,Eyeglass,0.03235,0.325780,0.110598,0.198102
5,1381,Male,Old,No Glasses,0.06905,0.677625,0.345250,0.209751
6,1339,Female,Young,Eyeglass,0.06695,0.674220,0.333085,0.196565
7,657,Female,Old,No Glasses,0.03285,0.322375,0.107178,0.196824


Saving to  /Users/adam2392/pytorch_data/CausalCelebA/eyeglass/dim128/obs/causalceleba_obs_joint_probability.csv

Pairwise Cramér's V Correlation Matrix:


...

Saving to  /Users/adam2392/pytorch_data/CausalCelebA/eyeglass/dim128/int_eye_0/causalceleba_int_eye_0_joint_probability.csv

Pairwise Cramér's V Correlation Matrix:


,Gender,Age,Eyeglasses
Gender,1.000000,0.339432,0.282544
Age,0.339432,1.000000,0.797331
Eyeglasses,0.282544,0.797331,1.000000


Conditional Cramér's V:
Eyeglasses = Eyeglass: Cramér's V = 0.1983
Eyeglasses = No Glasses: Cramér's V = 0.1956
Conditional Cramér's V:
Gender = Female: Cramér's V = 0.7784
Gender = Male: Cramér's V = 0.7761
Conditional Cramér's V:
Age = Old: Cramér's V = 0.0159
Age = Young: Cramér's V = 0.0251
[('Female', 'Young', 'Eyeglass'), ('Female', 'Old', 'No Glasses'), ('Male', 'Young', 'Eyeglass'), ('Female', 'Young', 'Eyeglass'), ('Male', 'Old', 'No Glasses')]


,Count,Gender,Age,Eye Glasses,Joint Probability,"P(Gender | Age, Eye Glasses)","P(Age | Gender, Eye Glasses)","P(Eye Glasses | Gender, Age)"
0,6135,Female,Young,Eyeglass,0.30675,0.680004,0.946175,0.900617
1,2989,Female,Old,No Glasses,0.14945,0.333892,0.815330,0.895446
2,2887,Male,Young,Eyeglass,0.14435,0.319996,0.822976,0.883956
3,5963,Male,Old,No Glasses,0.29815,0.666108,0.940240,0.905680
4,677,Female,Young,No Glasses,0.03385,0.641098,0.184670,0.099383
5,379,Male,Young,No Glasses,0.01895,0.358902,0.059760,0.116044
6,621,Male,Old,Eyeglass,0.03105,0.640206,0.177024,0.094320
7,349,Female,Old,Eyeglass,0.01745,0.359794,0.053825,0.104554


Saving to  /Users/adam2392/pytorch_data/CausalCelebA/eyeglass/dim128/int_eye_1/causalceleba_int_eye_1_joint_probability.csv
